# 02. Site-wide detection, measurement and classification

One pass over the 76 orthomosaic tiles: match each 2,500 px quadrant against the annotated set, run YOLOv8s-seg, measure every mask with `volume_pipeline`, then classify with the rules in `configs/thresholds.json`.

**Colour order.** Rasterio returns RGB; Ultralytics expects BGR arrays. Each quadrant goes through `cwd.inference.to_model_input` before `model.predict`. The September 2026 run omitted this and is superseded (see CHANGELOG.md).

Needs the project data root on Google Drive, which is not released. Run top to bottom on a T4 GPU, about 20 minutes. Completed tiles are skipped on a rerun. Output goes to `04. Results/site_run_v3/`.

In [ ]:
!pip -q install ultralytics pycocotools scikit-image rasterio
!test -d fallen-tree-detection || git clone -q https://github.com/AI-UGA/fallen-tree-detection.git

## Step 0. Config

In [ ]:
import os, sys, json, time, glob, gc, datetime, importlib
from dataclasses import asdict
import numpy as np, pandas as pd, cv2, torch

from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

sys.path.insert(0, os.path.abspath('fallen-tree-detection/src'))
from cwd.paths import require_root
from cwd.inference import to_model_input
from cwd.classify import verify_against_csv

ROOT = str(require_root())
print('ROOT:', ROOT)

# ------------------------------------------------------- resolved paths ---
TILE_DIR = f'{ROOT}/01. Dataset/Orthomosaic_Tiles'
DATASET  = f'{ROOT}/01. Dataset/DataV4/datav4.coco-segmentation'
OLD_DIR  = f'{ROOT}/04. Results/site_run'        # September run, read only
OUT_DIR  = f'{ROOT}/04. Results/site_run_v3'     # this run
PART_DIR = f'{OUT_DIR}/per_tile'
os.makedirs(PART_DIR, exist_ok=True)

# Fixed to the path recorded in site_run/...config.json
WEIGHTS = f'{ROOT}/04. Results/YOLOv8/best.pt'
assert os.path.exists(WEIGHTS), WEIGHTS

_vp = glob.glob(f'{ROOT}/*/volume_pipeline.py') + glob.glob(f'{ROOT}/*/*/volume_pipeline.py')
assert _vp, 'volume_pipeline.py not found'
PIPELINE = _vp[0]

# ------------------------------------------------------------ constants ---
GSD      = 0.0508     # m per pixel, GeoTIFF tag 33550
IMGSZ    = 1600
CONF     = 0.225      # F1 maximum on the validation split, box IoU 0.5
NMS_IOU  = 0.6
MAX_DET  = 700
QUAD_PX  = 2500

MATCH_MIN    = 0.99   # final value from the September rematch cell
MATCH_MARGIN = 0.05

EXPECTED_TILES = 76
EXPECTED_ANNOTATED = 209
EXPECTED_HA = 478.7

for label, p in [('tiles', TILE_DIR), ('dataset', DATASET), ('weights', WEIGHTS),
                 ('volume_pipeline.py', PIPELINE), ('old run', OLD_DIR), ('output', OUT_DIR)]:
    print('%-5s %-20s %s' % ('OK' if os.path.exists(p) else 'MISS', label, p))

assert OUT_DIR.endswith('site_run_v3'), 'refusing to write into the September run'
print('\ncuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## Step 1. Preflight on the rasters

In [ ]:
import rasterio

tifs = sorted(glob.glob(f'{TILE_DIR}/*.tif')) + sorted(glob.glob(f'{TILE_DIR}/*.tiff'))
print(f'{len(tifs)} tiles found, expected {EXPECTED_TILES}')

sizes, gsds, total_px = set(), set(), 0
for p in tifs:
    with rasterio.open(p) as s:
        sizes.add((s.width, s.height))
        gsds.add(round(abs(s.transform.a), 6))
        total_px += s.width * s.height
imaged_ha = total_px * GSD ** 2 / 10_000
print('dimensions :', sorted(sizes))
print('GSD m/px   :', gsds)
print(f'imaged     : {imaged_ha:.1f} ha')

assert len(tifs) == EXPECTED_TILES
assert len(gsds) == 1 and abs(list(gsds)[0] - GSD) < 1e-4
assert abs(imaged_ha - EXPECTED_HA) < 1.0
print('\npreflight passed')

## Step 2. Signatures of the 209 annotated quadrants (unchanged)

In [ ]:
def sig(img, k=32):
    g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if img.ndim == 3 else img
    s = cv2.resize(g, (k, k), interpolation=cv2.INTER_AREA).astype(np.float32)
    s -= s.mean()
    return (s / (np.linalg.norm(s) + 1e-9)).ravel()

ann_files, ann_split = [], []
for split in ('train', 'valid', 'test'):
    d_ = f'{DATASET}/{split}'
    fs = sorted(f for f in os.listdir(d_)
                if f.lower().endswith(('.jpg', '.jpeg', '.png')))
    ann_files += [f'{d_}/{f}' for f in fs]
    ann_split += [split] * len(fs)

print(f'{len(ann_files)} annotated images, expected {EXPECTED_ANNOTATED}')
assert len(ann_files) == EXPECTED_ANNOTATED

A, keep_split, keep_name = [], [], []
for p, sp in zip(ann_files, ann_split):
    im = cv2.imread(p)
    if im is None:
        print('unreadable:', p)
        continue
    A.append(sig(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)))
    keep_split.append(sp)
    keep_name.append(os.path.basename(p))
A = np.stack(A)
print(f'{A.shape[0]} signatures built')

## Step 3. Measurement function (unchanged)

In [ ]:
sys.path.insert(0, os.path.dirname(PIPELINE))
import volume_pipeline as vp
importlib.reload(vp)
from pycocotools import mask as coco_mask
from ultralytics import YOLO

def measure_one(rle, bbox, score, image_key, tile_id, offset_xy, tree_id,
                transform, tile_w, tile_h, was_annotated, match_corr):
    pred = {'segmentation': rle, 'bbox': list(map(float, bbox)),
            'score': float(score), 'image_id': image_key}
    m = asdict(vp.measure_tree(pred, tree_id, str(tile_id), GSD))

    ox, oy = offset_xy
    x, y, w, h = bbox
    gx1, gy1, gx2, gy2 = x + ox, y + oy, x + w + ox, y + h + oy
    cx, cy = (gx1 + gx2) / 2, (gy1 + gy2) / 2
    easting, northing = transform * (cx, cy)

    m['source_tile']       = str(tile_id)
    m['quadrant']          = image_key
    m['centroid_x_px']     = float(cx)
    m['centroid_y_px']     = float(cy)
    m['centroid_easting']  = float(easting)
    m['centroid_northing'] = float(northing)
    m['mask_area_m2']      = m['mask_area_px'] * GSD ** 2
    m['touches_tile_edge'] = bool(gx1 <= 1 or gy1 <= 1 or
                                  gx2 >= tile_w - 1 or gy2 >= tile_h - 1)
    m['was_annotated']     = bool(was_annotated)
    m['match_corr']        = float(match_corr)
    return m

print('measure_one ready, pipeline from', vp.__file__)

## Step 4. The run

Set `LIMIT = 1` for a smoke test, then back to `None`. Completed tiles are skipped.

Inference input goes through `to_model_input`.

In [ ]:
LIMIT = None      # set to 1 for a smoke test on one tile

assert OUT_DIR.endswith('site_run_v3')
model = YOLO(WEIGHTS)
print('weights loaded, task', model.task, 'classes', model.names)

def quadrants(w, h, q=QUAD_PX):
    """Yield (key, x0, y0, x1, y1). Clipped tiles give partial quadrants."""
    for r, y0 in enumerate(range(0, h, q)):
        for c, x0 in enumerate(range(0, w, q)):
            yield f'q{r}{c}', x0, y0, min(x0 + q, w), min(y0 + q, h)

todo = tifs if LIMIT is None else tifs[:LIMIT]
t_start = time.time()

for ti, tif in enumerate(todo, 1):
    tile_id = os.path.splitext(os.path.basename(tif))[0]
    part  = f'{PART_DIR}/{tile_id}.csv'
    qpart = f'{PART_DIR}/{tile_id}.quads.csv'
    if os.path.exists(part) and os.path.exists(qpart):
        print(f'[{ti}/{len(todo)}] {tile_id} already done, skipped')
        continue

    with rasterio.open(tif) as src:
        tw, th = src.width, src.height
        transform = src.transform
        nb = min(3, src.count)
        arr = src.read(indexes=list(range(1, nb + 1)))
    img = np.transpose(arr, (1, 2, 0))          # RGB
    if img.shape[2] == 1:
        img = np.repeat(img, 3, axis=2)
    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)
    del arr

    rows, qrows = [], []
    for qkey, x0, y0, x1, y1 in quadrants(tw, th):
        sub = img[y0:y1, x0:x1]                  # RGB, used for provenance
        if sub.shape[0] < 64 or sub.shape[1] < 64:
            continue

        # provenance match against the annotated set (RGB, unchanged)
        v = sig(sub)
        corr = A @ v
        order = np.argsort(-corr)
        best = int(order[0])
        second = int(order[1]) if len(order) > 1 else best
        c1, c2 = float(corr[best]), float(corr[second])
        is_ann = (c1 >= MATCH_MIN) and (c1 - c2 >= MATCH_MARGIN)

        qrows.append({'source_tile': tile_id, 'quadrant': qkey,
                      'x0': x0, 'y0': y0,
                      'w_px': x1 - x0, 'h_px': y1 - y0,
                      'area_ha': (x1 - x0) * (y1 - y0) * GSD ** 2 / 10_000,
                      'was_annotated': bool(is_ann),
                      'match_index': best if is_ann else -1,
                      'match_file': keep_name[best] if is_ann else '',
                      'match_split': keep_split[best] if is_ann else '',
                      'corr': c1, 'corr_runner_up': c2})

        # inference
        sub_bgr = to_model_input(sub)                    # Ultralytics expects BGR arrays
        r = model.predict(sub_bgr, imgsz=IMGSZ, conf=CONF, iou=NMS_IOU,
                          max_det=MAX_DET, retina_masks=False,
                          augment=False, verbose=False)[0]
        if r.masks is not None:
            boxes  = r.boxes.xyxy.cpu().numpy()
            scores = r.boxes.conf.cpu().numpy()
            qh, qw = sub.shape[:2]
            for k in range(len(r.masks.data)):
                mk = r.masks.data[k].cpu().numpy().astype(np.uint8)
                if mk.shape != (qh, qw):
                    mk = cv2.resize(mk, (qw, qh), interpolation=cv2.INTER_NEAREST)
                if mk.sum() == 0:
                    del mk
                    continue
                enc = coco_mask.encode(np.asfortranarray(mk))
                rle = {'counts': enc['counts'].decode('ascii'),
                       'size': list(enc['size'])}
                bx1, by1, bx2, by2 = boxes[k]
                bbox = [float(bx1), float(by1), float(bx2 - bx1), float(by2 - by1)]
                try:
                    rows.append(measure_one(
                        rle, bbox, scores[k], qkey, tile_id, (x0, y0),
                        f'{tile_id}_{qkey}_{k:04d}', transform, tw, th,
                        is_ann, c1))
                except Exception as e:
                    print('   measure failed', tile_id, qkey, k, type(e).__name__)
                del mk
        del r, sub_bgr
        gc.collect()

    pd.DataFrame(qrows).to_csv(qpart, index=False)
    pd.DataFrame(rows).to_csv(part, index=False)
    del img
    gc.collect()

    el = time.time() - t_start
    print(f'[{ti}/{len(todo)}] {tile_id}  {tw}x{th}  {len(qrows)} quadrants  '
          f'{len(rows):,} objects  elapsed {el/60:.1f} min  '
          f'eta {el/ti*(len(todo)-ti)/60:.0f} min')

print('\ndone')

## Step 5. Provenance check (expect 202 matched, no collisions)

In [ ]:
Q = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(f'{PART_DIR}/*.quads.csv'))],
              ignore_index=True)
Q['qkey'] = Q.source_tile + '/' + Q.quadrant
ann = Q[Q.was_annotated]

print(f'quadrants total          {len(Q)}  (expect 300)')
print(f'flagged annotated        {len(ann)}  (expect 202)')
print(f'distinct images claimed  {ann.match_index.nunique()}')
dup = ann.groupby('match_index').size()
print(f'images claimed twice     {int((dup > 1).sum())}  (expect 0)')

assert len(Q) == 300
assert len(ann) == ann.match_index.nunique(), 'collisions remain'

matched = {int(q.match_index): (q.source_tile, q.quadrant, float(q['corr']))
           for _, q in ann.iterrows()}
m = pd.DataFrame([{'file_name': keep_name[j], 'split': keep_split[j],
                   'tile': v[0], 'quadrant': v[1], 'corr': v[2]}
                  for j, v in matched.items()])
m.to_csv(f'{OUT_DIR}/annotated_to_tile_map.csv', index=False)

# area split must be identical to September, since provenance is unchanged
print('\narea by annotation status (expect 152.929 and 325.806 ha)')
print(Q.groupby('was_annotated').area_ha.sum().round(3).to_string())

## Step 6. Assemble and check

In [ ]:
parts = sorted(p for p in glob.glob(f'{PART_DIR}/*.csv') if not p.endswith('.quads.csv'))
df = pd.concat([pd.read_csv(p) for p in parts if os.path.getsize(p) > 5], ignore_index=True)
df['qkey'] = df.source_tile + '/' + df.quadrant

csv_path = f'{OUT_DIR}/cwd_measurements_site_gsd{GSD}_conf{CONF:.3f}.csv'
df.to_csv(csv_path, index=False)

import skimage, scipy, ultralytics
json.dump({
    'written': datetime.datetime.now().isoformat(timespec='seconds'),
    'mode': 'site', 'root': ROOT,
    'colour_order': 'RGB raster converted to BGR before model.predict (fixed in v3)',
    'supersedes': f'{OLD_DIR} (RGB array passed as BGR)',
    'gsd_m_per_px': GSD, 'operating_confidence': CONF,
    'threshold_selection': 'F1 maximum on the 42 image validation split, box IoU 0.5',
    'imgsz': IMGSZ, 'max_det': MAX_DET, 'nms_iou': NMS_IOU,
    'retina_masks': False, 'match_min': MATCH_MIN, 'match_margin': MATCH_MARGIN,
    'width_sample_interval_px': vp.WIDTH_SAMPLE_INTERVAL_PX,
    'weights': WEIGHTS, 'pipeline': PIPELINE,
    'tiles': len(parts), 'n_objects': int(len(df)),
    'quadrants': int(len(Q)), 'imaged_ha': float(Q.area_ha.sum()),
    'annotated_quadrants_matched': len(matched),
    'versions': {'numpy': np.__version__, 'scipy': scipy.__version__,
                 'skimage': skimage.__version__, 'pandas': pd.__version__,
                 'ultralytics': ultralytics.__version__, 'torch': torch.__version__},
}, open(csv_path.replace('.csv', '.config.json'), 'w'), indent=2)

area_ha = Q.area_ha.sum()
print(f'{len(df):,} objects across {len(parts)} tiles')
print(f'{area_ha:.1f} ha, total volume {df.volume_m3.sum():,.1f} m3')
print('saved', csv_path)

## Step 7. Classification and Tables 8 and 9

Same rule set as September. `L_FLOOR` and `L_STEM` are frozen at the September values.

In [ ]:
d = pd.read_csv(csv_path)
d['touches_tile_edge'] = d.touches_tile_edge.astype(str).str.lower().eq('true')

WIDTH_BIAS   = 1.19        # median predicted / annotated width, 1,605 pairs
STEM_PCTL    = 75          # how L_STEM was chosen in September, kept for the record
E_STEM       = 12.0
W_MIN_STEM   = 0.25
W_MAX_STEM   = 1.50
W_FLOOR      = 0.15        # 3.0 px
E_FLOOR      = 3.0
L_FLOOR      = 1.37        # frozen, September p05
L_STEM       = 6.78        # frozen, September p75

d['width_corr_m'] = d.width_mean_m / WIDTH_BIAS
d['elongation']   = (d.length_m / d.width_corr_m.replace(0, np.nan)).fillna(0)

print(f'for reference, this run would have given p05 {d.length_m.quantile(.05):.2f} m '
      f'and p75 {d.length_m.quantile(.75):.2f} m; frozen values used instead')

cond = [
    d.width_corr_m < W_FLOOR,
    d.length_m     < L_FLOOR,
    d.elongation   < E_FLOOR,
    d.touches_tile_edge,
    (d.length_m >= L_STEM) & (d.elongation >= E_STEM)
        & d.width_corr_m.between(W_MIN_STEM, W_MAX_STEM),
]
choice = ['Excluded, below width threshold', 'Excluded, below length threshold',
          'Excluded, insufficient elongation', 'Excluded, tile boundary contact',
          'Full stem']
d['klass'] = np.select(cond, choice, default='Fragment')

ORDER = ['Full stem', 'Fragment',
         'Excluded, below length threshold', 'Excluded, below width threshold',
         'Excluded, insufficient elongation', 'Excluded, tile boundary contact',
         'Excluded, outside forest boundary']

t8 = (d.klass.value_counts().reindex(ORDER).fillna(0).astype(int)
      .rename('objects').to_frame())
t8['percent'] = (t8.objects / len(d) * 100).round(1)
vol = d.groupby('klass').volume_m3.sum().reindex(ORDER).fillna(0)
t8['volume_m3'] = vol.round(1)
t8['m3_per_ha'] = (vol / area_ha).round(2)
t8.loc['Total detected'] = [len(d), 100.0, round(d.volume_m3.sum(), 1),
                            round(d.volume_m3.sum() / area_ha, 2)]
counted = d[d.klass.isin(['Full stem', 'Fragment'])]
t8.loc['Inventory total'] = [len(counted), round(len(counted) / len(d) * 100, 1),
                             round(counted.volume_m3.sum(), 1),
                             round(counted.volume_m3.sum() / area_ha, 2)]
print('\n=== Table 8. Inventory composition (v3) ===')
print(t8.to_string())

def col(g):
    return pd.Series({
        'Objects measured': len(g),
        'Mean volume, m3': g.volume_m3.mean(),
        'Median volume, m3': g.volume_m3.median(),
        'Standard deviation': g.volume_m3.std(),
        'Minimum': g.volume_m3.min(),
        'Maximum': g.volume_m3.max(),
        'Total volume, m3': g.volume_m3.sum(),
        'Volume per hectare': g.volume_m3.sum() / area_ha,
        'Median length, m': g.length_m.median(),
        'Median width, m': g.width_mean_m.median()})

t9 = pd.DataFrame({
    'Full stems': col(counted[counted.klass == 'Full stem']),
    'Fragments':  col(counted[counted.klass == 'Fragment']),
    'Combined':   col(counted)}).round(3)
print('\n=== Table 9. Volume descriptive statistics by class (v3) ===')
print(t9.to_string())

# densities by annotation status, inventory basis (paper headline)
qa = Q.groupby('was_annotated').area_ha.sum()
flag = dict(zip(Q.qkey, Q.was_annotated))
d['quad_annotated'] = (d.source_tile + '/' + d.quadrant).map(flag)
inv = d[d.klass.isin(['Full stem', 'Fragment'])]
print('\ninventory basis by annotation status')
for f_, label in [(False, 'unannotated'), (True, 'annotated')]:
    s = inv[inv.quad_annotated == f_]
    print('%-12s %6s objects  %10s m3  %6.2f m3/ha'
          % (label, format(len(s), ','), format(s.volume_m3.sum(), ',.1f'),
             s.volume_m3.sum() / qa[f_]))

tot = counted.volume_m3.sum()
print(f'\ninventory   {len(counted):,} objects   {tot:,.1f} m3   '
      f'{tot/area_ha:.2f} m3/ha')

t8.to_csv(f'{OUT_DIR}/table8_composition.csv')
t9.to_csv(f'{OUT_DIR}/table9_volume_by_class.csv')
d.drop(columns=['quad_annotated']).to_csv(f'{OUT_DIR}/cwd_classified_site.csv', index=False)
json.dump({'WIDTH_BIAS': WIDTH_BIAS, 'width_correction_applies_to': 'classification only',
           'L_STEM': L_STEM, 'L_STEM_source': 'frozen from September run (p75)',
           'E_STEM': E_STEM, 'W_MIN_STEM': W_MIN_STEM, 'W_MAX_STEM': W_MAX_STEM,
           'L_FLOOR': L_FLOOR, 'L_FLOOR_source': 'frozen from September run (p05)',
           'W_FLOOR': W_FLOOR, 'W_FLOOR_px': W_FLOOR / GSD, 'E_FLOOR': E_FLOOR,
           'forest_boundary_applied': False,
           'source_csv': csv_path, 'area_ha': float(area_ha),
           'n_detected': int(len(d)), 'n_inventory': int(len(counted)),
           'total_volume_m3': float(tot), 'm3_per_ha': float(tot / area_ha)},
          open(f'{OUT_DIR}/table7_thresholds.json', 'w'), indent=2)
print('\nsaved tables 8 and 9, thresholds and cwd_classified_site.csv to', OUT_DIR)

## Step 8. Verify against the published counts

In [ ]:
summary = verify_against_csv(f'{OUT_DIR}/cwd_classified_site.csv')
print('rows %d  agreement %.4f' % (summary['rows'], summary['agreement']))
for k, v in sorted(summary['counts'].items()):
    print('  %-36s %7d' % (k, v))

## Step 9. Optional: compare with the superseded September run

In [ ]:
if not os.path.exists(f'{OLD_DIR}/table8_composition.csv'):
    raise SystemExit('no September run to compare with')
old = pd.read_csv(f'{OLD_DIR}/table8_composition.csv', index_col=0)
new = pd.read_csv(f'{OUT_DIR}/table8_composition.csv', index_col=0)
cmp = pd.DataFrame({'sept objects': old.objects, 'v3 objects': new.objects,
                    'sept m3': old.volume_m3, 'v3 m3': new.volume_m3,
                    'sept m3/ha': old.m3_per_ha, 'v3 m3/ha': new.m3_per_ha})
cmp['volume change %'] = ((cmp['v3 m3'] / cmp['sept m3'].replace(0, np.nan) - 1) * 100).round(1)
print(cmp.to_string())